In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model
from tabulate import tabulate
import os

# Your stocks and their company names
stocks = [
    ('AAPL', 'Apple Inc.'),
    ('MSFT', 'Microsoft Corporation'),
    ('GOOGL', 'Alphabet Inc.'),
    ('AMZN', 'Amazon.com, Inc.'),
    ('NVDA', 'NVIDIA Corporation'),
    ('META', 'Meta Platforms, Inc.'),
    ('TSLA', 'Tesla, Inc.'),
    ('BRK-B', 'Berkshire Hathaway Inc.'),
    ('UNH', 'UnitedHealth Group Incorporated'),
    ('JPM', 'JPMorgan Chase & Co.')
]

# Prepare table headers
headers = ["Rank", "Stock Symbol", "Company Name"]

# Add rank to data
table_data = [(i+1, symbol, name) for i, (symbol, name) in enumerate(stocks)]

# Print the table
print(tabulate(table_data, headers=headers, tablefmt="grid"))


# === Step 1: User Inputs ===
symbol = input("Enter stock symbol (e.g., AAPL): ").upper()
time_steps = int(input("Enter number of lookback days (e.g., 60, 120, 365): "))
predict_date_input = input("Enter prediction date (YYYY-MM-DD) or leave blank for next trading day: ")

predict_date = predict_date_input if predict_date_input else None

# === Step 2: Load model ===
model_path = f"models/{symbol}_best_model.h5"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Trained model for {symbol} not found at {model_path}.")

model = load_model(model_path)

# === Step 3: Download recent stock data ===
end_date = datetime.today().strftime('%Y-%m-%d') if predict_date is None else predict_date
start_date = (datetime.strptime(end_date, '%Y-%m-%d') - timedelta(days=time_steps * 2)).strftime('%Y-%m-%d')

df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
df = df[['Date', 'Close']].dropna()

if len(df) < time_steps:
    raise ValueError(f"Not enough data to form {time_steps} time steps. Only {len(df)} data points available.")

# === Step 4: Prepare input for prediction ===
scaler = MinMaxScaler()
scaled_close = scaler.fit_transform(df[['Close']].values)

last_sequence = scaled_close[-time_steps:]
last_sequence = np.expand_dims(last_sequence, axis=0)  # shape: (1, time_steps, 1)

# === Step 5: Predict ===
scaled_prediction = model.predict(last_sequence)
predicted_price = scaler.inverse_transform(scaled_prediction)[0][0]

# === Step 6: Display Results ===
target_day = "next trading day" if predict_date is None else predict_date
print(f"\n📈 Predicted closing price for {symbol} on {target_day} (using last {time_steps} days): ${predicted_price:.2f}")


+--------+----------------+---------------------------------+
|   Rank | Stock Symbol   | Company Name                    |
+========+================+=================================+
|      1 | AAPL           | Apple Inc.                      |
+--------+----------------+---------------------------------+
|      2 | MSFT           | Microsoft Corporation           |
+--------+----------------+---------------------------------+
|      3 | GOOGL          | Alphabet Inc.                   |
+--------+----------------+---------------------------------+
|      4 | AMZN           | Amazon.com, Inc.                |
+--------+----------------+---------------------------------+
|      5 | NVDA           | NVIDIA Corporation              |
+--------+----------------+---------------------------------+
|      6 | META           | Meta Platforms, Inc.            |
+--------+----------------+---------------------------------+
|      7 | TSLA           | Tesla, Inc.                     |
+-------

/tmp/ipykernel_62340/804377776.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step

📈 Predicted closing price for TSLA on next trading day (using last 60 days): $312.20
